In [ ]:
# Install required libraries
!pip -q install pandas numpy scikit-learn transformers datasets accelerate torch tqdm

In [ ]:
# Imports and reproducibility
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


In [ ]:
# Mount Google Drive and define paths
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/multisocial_outputs'
TRAIN_CSV = os.path.join(BASE_DIR, 'multisocial_train.csv')
TEST_CSV = os.path.join(BASE_DIR, 'multisocial_test.csv')
OUTPUT_BASE = os.path.join(BASE_DIR, 'transfer_xlmr')
os.makedirs(OUTPUT_BASE, exist_ok=True)

# CSV output directories
RESULTS_DIR = '/content/results'
DRIVE_RESULTS_DIR = OUTPUT_BASE
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

MODEL_NAME = 'xlm-roberta-base'
MAX_LENGTH = 512
BATCH_SIZE = 16
NUM_EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

assert os.path.exists(TRAIN_CSV), f'Train file not found: {TRAIN_CSV}'
assert os.path.exists(TEST_CSV), f'Test file not found: {TEST_CSV}'
print(f'Train: {TRAIN_CSV}')
print(f'Test: {TEST_CSV}')
print(f'Output: {OUTPUT_BASE}')

Mounted at /content/drive
Train: /content/drive/MyDrive/multisocial_outputs/multisocial_train.csv
Test: /content/drive/MyDrive/multisocial_outputs/multisocial_test.csv
Output: /content/drive/MyDrive/multisocial_outputs/transfer_xlmr


In [ ]:
# Load and validate data
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

required_cols = {'text', 'label', 'language'}
for name, df in [('train', train_df), ('test', test_df)]:
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns in {name} CSV: {sorted(missing)}')

train_df = train_df.dropna(subset=['text', 'label', 'language']).copy()
test_df = test_df.dropna(subset=['text', 'label', 'language']).copy()
train_df['text'] = train_df['text'].astype(str)
train_df['label'] = train_df['label'].astype(int)
train_df['language'] = train_df['language'].astype(str)
test_df['text'] = test_df['text'].astype(str)
test_df['label'] = test_df['label'].astype(int)
test_df['language'] = test_df['language'].astype(str)

print(f'Train: {len(train_df)} rows')
print(f'Test: {len(test_df)} rows')
print('\nTrain per language:')
print(train_df['language'].value_counts())
print('\nTest per language:')
print(test_df['language'].value_counts())

Train: 12789 rows
Test: 3197 rows

Train per language:
language
en    3200
zh    3200
ar    3200
vi    3189
Name: count, dtype: int64

Test per language:
language
en    800
zh    800
ar    800
vi    797
Name: count, dtype: int64


In [ ]:
# Helper functions
def safe_roc_auc(y_true, y_score):
    """Compute ROC-AUC; return NaN if single-class."""
    if len(np.unique(y_true)) < 2:
        print('  Warning: single class present — AUC undefined, returning NaN')
        return float('nan')
    return float(roc_auc_score(y_true, y_score))

def compute_classification_metrics(y_true, y_pred, y_prob):
    """Compute 5-metric classification report."""
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'roc_auc': safe_roc_auc(y_true, y_prob),
    }

def save_df(df, basename):
    """Save DataFrame as CSV to both local and Drive directories."""
    local_path = os.path.join(RESULTS_DIR, basename)
    drive_path = os.path.join(DRIVE_RESULTS_DIR, basename)
    df.to_csv(local_path, index=False)
    df.to_csv(drive_path, index=False)
    print(f'  Saved: {local_path}')
    print(f'  Saved: {drive_path}')

print('Helper functions defined.')

Helper functions defined.


In [ ]:
# Define and run transfer experiments
LANGUAGES = ['en', 'vi', 'zh', 'ar']

def train_and_evaluate(train_langs, test_lang, train_df, test_df):
    """Train on train_langs, evaluate on test_lang."""
    print(f'\n{"="*60}')
    print(f'TRAIN on {train_langs} -> TEST on {test_lang}')
    print(f'{"="*60}')

    # Split train into train/val (80/20 stratified)
    subset = train_df[train_df['language'].isin(train_langs)].copy()
    subset = subset.sample(frac=1, random_state=SEED).reset_index(drop=True)
    split_idx = int(len(subset) * 0.8)
    train_split = subset.iloc[:split_idx]
    val_split = subset.iloc[split_idx:]

    test_subset = test_df[test_df['language'] == test_lang].copy()

    print(f'  Train: {len(train_split)} | Val: {len(val_split)} | Test: {len(test_subset)}')
    print(f'  Train label dist: {dict(train_split["label"].value_counts())}')
    print(f'  Test label dist: {dict(test_subset["label"].value_counts())}')

    # Tokenize
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_fn(examples):
        return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=MAX_LENGTH)

    train_ds = Dataset.from_pandas(train_split[['text', 'label']]).map(tokenize_fn, batched=True)
    val_ds = Dataset.from_pandas(val_split[['text', 'label']]).map(tokenize_fn, batched=True)
    test_ds = Dataset.from_pandas(test_subset[['text', 'label']]).map(tokenize_fn, batched=True)

    train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    val_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

    # Load model
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model.config.problem_type = 'single_label_classification'

    # Metrics
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
        preds = np.argmax(logits, axis=-1)
        return compute_classification_metrics(labels, preds, probs)

    # Training
    ckpt_dir = os.path.join(OUTPUT_BASE, f'checkpoints_{test_lang}')
    training_args = TrainingArguments(
        output_dir=ckpt_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        logging_steps=50,
        seed=SEED,
        fp16=torch.cuda.is_available(),
        report_to='none',
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    # Evaluate
    eval_output = trainer.predict(test_ds)
    logits = eval_output.predictions
    labels = eval_output.label_ids
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = np.argmax(logits, axis=-1)

    metrics = compute_classification_metrics(labels, preds, probs)
    cm = confusion_matrix(labels, preds)

    print(f'\n  Results for {test_lang}:')
    print(f'    Accuracy:  {metrics["accuracy"]:.4f}')
    print(f'    Precision: {metrics["precision"]:.4f}')
    print(f'    Recall:    {metrics["recall"]:.4f}')
    print(f'    F1:        {metrics["f1"]:.4f}')
    print(f'    ROC-AUC:   {metrics["roc_auc"]:.4f}')
    print(f'    Confusion matrix:\n{cm}')

    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

    return {
        'test_language': test_lang,
        'train_languages': train_langs,
        **metrics,
        'confusion_matrix': cm.tolist(),
    }

print('train_and_evaluate() defined.')

train_and_evaluate() defined.


In [ ]:
import sys
import datasets.config

# Explicitly disable torchvision check in datasets to prevent VideoReader ImportError
datasets.config.TORCHVISION_AVAILABLE = False

# Run all 4 transfer experiments
results = []
for held_out in LANGUAGES:
    train_langs = [l for l in LANGUAGES if l != held_out]
    result = train_and_evaluate(train_langs, held_out, train_df, test_df)
    results.append(result)

print(f'\nAll {len(results)} experiments complete.')


TRAIN on ['vi', 'zh', 'ar'] -> TEST on en
  Train: 7671 | Val: 1918 | Test: 800
  Train label dist: {1: np.int64(3846), 0: np.int64(3825)}
  Test label dist: {0: np.int64(400), 1: np.int64(400)}


Map:   0%|          | 0/7671 [00:00<?, ? examples/s]

Map:   0%|          | 0/1918 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Results for ar:
    Accuracy:  0.6188
    Precision: 0.5809
    Recall:    0.8525
    F1:        0.6910
    ROC-AUC:   0.8101
    Confusion matrix:
[[154 246]
 [ 59 341]]

All 4 experiments complete.


In [ ]:
# Summarize results
results_df = pd.DataFrame(results)
print('Transfer experiment results:')
display(results_df[['test_language', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']])

overall_acc = results_df['accuracy'].mean()
overall_f1 = results_df['f1'].mean()
overall_auc = results_df['roc_auc'].mean()
print(f'\nMacro-average — Accuracy: {overall_acc:.4f} | F1: {overall_f1:.4f} | ROC-AUC: {overall_auc:.4f}')

Transfer experiment results:


,test_language,accuracy,precision,recall,f1,roc_auc
0,en,0.54500,0.524064,0.980000,0.682927,0.850725
1,vi,0.53074,0.568627,0.076115,0.134259,0.536121
2,zh,0.50250,0.501266,0.990000,0.665546,0.512428
3,ar,0.61875,0.580920,0.852500,0.690983,0.810131



Macro-average — Accuracy: 0.5492 | F1: 0.5434 | ROC-AUC: 0.6774


In [ ]:
# Save CSV exports
print('Saving transfer results as CSV...')

# 1) Per-language transfer results
save_df(results_df, 'transfer_per_language.csv')

# 2) Overall summary
summary = pd.DataFrame([{
    'model': MODEL_NAME,
    'num_experiments': len(results),
    'mean_accuracy': overall_acc,
    'mean_precision': results_df['precision'].mean(),
    'mean_recall': results_df['recall'].mean(),
    'mean_f1': overall_f1,
    'mean_roc_auc': overall_auc,
    'train_rows': int(len(train_df)),
    'test_rows': int(len(test_df)),
}])
save_df(summary, 'transfer_overall.csv')

print('CSV export complete.')

Saving transfer results as CSV...
  Saved: /content/results/transfer_per_language.csv
  Saved: /content/drive/MyDrive/multisocial_outputs/transfer_xlmr/transfer_per_language.csv
  Saved: /content/results/transfer_overall.csv
  Saved: /content/drive/MyDrive/multisocial_outputs/transfer_xlmr/transfer_overall.csv
CSV export complete.


In [ ]:
# Save JSON results
RESULTS_JSON = os.path.join(OUTPUT_BASE, 'results_transfer.json')

payload = {
    'seed': SEED,
    'model_name': MODEL_NAME,
    'num_experiments': len(results),
    'mean_accuracy': float(overall_acc),
    'mean_f1': float(overall_f1),
    'mean_roc_auc': float(overall_auc),
    'per_language': results_df.to_dict(orient='records'),
}

with open(RESULTS_JSON, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

assert os.path.exists(RESULTS_JSON), f'Results JSON not found: {RESULTS_JSON}'
print(f'Saved transfer results to: {RESULTS_JSON}')

Saved transfer results to: /content/drive/MyDrive/multisocial_outputs/transfer_xlmr/results_transfer.json
